# ViSpatialQA — E4: cải thiện nhánh LP (CPU) + cross-fit F2C để tune trên 613 câu Human (GPU)

**Phần 1 (CPU, khoảng 10 phút): LP+.** Dùng lại toàn bộ JSON Qwen3-32B đã sinh ở E3 (không chạy lại GPU) và thử thêm các công tắc sau:

| Ý tưởng | Công tắc | Nội dung |
|---|---|---|
| B1 | `forms_mode` | parser luật cho câu hỏi Human (đọc được 84% câu); `rule_first` = luật trước, LLM dự phòng; `rule_plus` = luật + LLM cùng bỏ phiếu |
| B2 | `relax` | mô tả không khớp vật nào → bỏ dần size, rồi color |
| C2 | `definite` | "X có … không" khi X khớp nhiều vật: `any` hay `all` |
| — | `fr_block_share` | "X và Y trong C": X không ghi khối thì lấy khối của Y |
| D1 | `world_merge` | gộp 4 world, giữ fact có trong ≥ 1/2 số mẫu |
| C1 | bảng gần/xa FR | học trên Human train khi nào gold thêm "gần"/"xa"; chấm **ngoài fold** để không lạc quan |

- Mọi lựa chọn được thực hiện trên **Human train (481 câu)**. Dev chỉ dùng để xem kết quả.
- Nguồn mới tạo ra là **LPX**.

**Phần 2 (GPU): A1, cross-fit.**
- F2C được train lại 5 lần. Mỗi lần bỏ ra 6/30 story Human train, rồi dự đoán đúng 6 story đó, nên dự đoán trên train là sạch.
- F1 và L (Qwen3-32B CoT) cũng chạy trên Human train.
- Kết quả: F, L và LP đều có dự đoán **ngoài mẫu** trên cả **613 câu (37 story)**, thay vì chỉ 116.

**Phần 3: so sánh.** Tune ensemble trên train + dev (cv 5-fold theo story) và so sánh ba biến thể: không LP, LP (E3), LPX.

**Cần có:** `outputs/predictions_e3` trên Drive (do notebook E3 tạo), adapter F1 (Drive `ckpt/f_auto` hoặc HF).

### Thời gian ước tính (H100/Blackwell ~80–96GB)
| Bước | Ước tính | Căn cứ |
|---|---|---|
| Chuẩn bị (pip vllm, clone, dữ liệu) | ~5 phút | các lần chạy trước: 2.4–2.8 phút cài đặt |
| Phần 1: LP+ (CPU) | 5–10 phút | grid 108 cấu hình × 481 câu, dựng world một lần mỗi story |
| F1 trên Human train | ~3 phút | infer_f lần trước: 1–3.5 phút/lần (chủ yếu nạp model) |
| L trên Human train (~376 câu còn thiếu YN/FB/CO) | 20–50 phút | L trước đây: 2.5–8.7 giây/câu với 8 mẫu |
| Cross-fit F2C: 5 × (chuẩn bị + train + dự đoán) | 55–65 phút | train stage 2 lần trước: 10.1 phút/lần với ~2.5k bản ghi × 3 epoch; mỗi fold ~4/5 số câu Human |
| Phần 3: ghép + so sánh 3 biến thể | 5–10 phút | compare lần trước 0.2–0.4 phút trên 116 câu; nay 613 câu |
| **Tổng** | **≈ 1 giờ 35 phút – 2 giờ 25 phút** | |

Có thể tắt từng phần trong CONFIG. Mọi bước đều resume được sau khi Colab ngắt, nhờ các file đã sao lưu lên Drive.

## 1. CONFIG

In [ ]:
CONFIG = dict(
    REPO_URL="https://github.com/lightanddark0/VLSP-Task5.git",
    BRANCH="feat/e4-lp-crossfit",
    WORK_DIR="/content/VLSP-Task5",
    DRIVE_DATA_DIR="/content/drive/MyDrive/VLSP2026/Data",
    DRIVE_BACKUP_DIR="/content/drive/MyDrive/VLSP2026/outputs",
    DRIVE_CKPT_DIR="/content/drive/MyDrive/VLSP2026/ckpt",
    HF_USER="",                                   # để trống = lấy từ token
    HF_F_AUTO_REPO="qwen3-8b-vi-spatial-lora-s1", # adapter F1 (dùng nếu Drive không có ckpt/f_auto)
    USE_WANDB=True, WANDB_PROJECT="vispatialqa-vlsp2026",
    F_MODEL="Qwen/Qwen3-8B",
    INFER_F=dict(max_model_len=1024, max_num_seqs=128, gpu_memory_utilization=0.90),
    LCOT=dict(model="Qwen/Qwen3-32B-FP8", fallback="Qwen/Qwen3-32B", n=8, max_tokens=8192, max_model_len=16384,
              max_num_seqs=64, fewshot_k=2, gpu_memory_utilization=0.90),
    # F2C giống E2: câu đảo chiều FR+YN, 2000 câu Auto, 3 epoch, lr 5e-5, chọn epoch theo eval_loss trên Human dev
    F2C=dict(augment="FR,YN", human_repeat=1, auto_mix=2000, epochs=3, learning_rate=5e-5,
             effective_batch=16, batch_sizes=[16, 8], max_len=1024, folds=5),
    LP_BASE=["cross_block_far=true"],             # quy ước E3 đã chọn trên Human train
    RUN_LP_PLUS=True, RUN_F1_TRAIN=True, RUN_LCOT_TRAIN=True, RUN_CROSSFIT=True, RUN_COMPARE=True,
    USE_INDIFINITE=False, HUMAN_YN_DK="keep",
    SUBMISSION_NAMES=dict(human="human_submission.json", auto="auto_submission.json"),
    SEED=42,
)

## 2. Chuẩn bị: code, thư viện, Drive, dữ liệu, dự đoán E3, HF/W&B

In [ ]:
import codecs, json, os, shlex, shutil, subprocess, sys, time
from pathlib import Path

C = CONFIG
WORK = Path(C["WORK_DIR"])
OUT = WORK / "outputs"
PRED = OUT / "predictions_e4"        # dự đoán của notebook này + bản sao từ E3
CKPT = Path(C["DRIVE_CKPT_DIR"])
DRIVE_OUT = Path(C["DRIVE_BACKUP_DIR"])
os.environ["VLLM_USE_DEEP_GEMM"] = "0"            # FP8 JIT cần NVCC >= 12.9
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"   # sampler JIT cần CUDA >= 12.9 trên GPU SM 12.x
START = time.time()


def sh(cmd, check=True):
    """Chạy lệnh (list hoặc str) trong WORK_DIR, in output trực tiếp (giữ thanh tiến trình)."""
    if isinstance(cmd, (list, tuple)):
        cmd = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ {cmd}", flush=True)
    started = time.time()
    process = subprocess.Popen(cmd, shell=True, cwd=WORK if WORK.exists() else None, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    while chunk := os.read(process.stdout.fileno(), 4096):
        sys.stdout.write(decoder.decode(chunk)); sys.stdout.flush()
    code = process.wait()
    print(f"\n[exit {code}, {(time.time() - started) / 60:.1f} phút | tổng {(time.time() - START) / 60:.1f} phút]")
    if check and code:
        raise RuntimeError(f"Lệnh lỗi (exit {code}): {cmd}")
    return code


def gpu(title=""):
    print(f"--- nvidia-smi {title}")
    subprocess.run("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv", shell=True)


def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""


def split_file(dataset, split):
    return {"train": f"Data/splits/{dataset}_train.json", "dev": f"Data/splits/{dataset}_dev.json",
            "test": f"Data/{dataset}_public_test.json"}[split]


def jobs(source, pairs):
    """--job arguments writing into outputs/predictions_e1e2/<source>/<dataset>_<split>.jsonl."""
    out = []
    for dataset, split in pairs:
        out += ["--job", f"{dataset},{split_file(dataset, split)},{PRED / source / f'{dataset}_{split}.jsonl'}"]
    return out


def to_drive(folder):
    if Path("/content/drive/MyDrive").exists() and (OUT / folder).exists():
        shutil.copytree(OUT / folder, DRIVE_OUT / folder, dirs_exist_ok=True)
        print(f"Đã sao lưu {OUT / folder} -> {DRIVE_OUT / folder}")


def train(cmd, output_dir, batch_sizes, effective_batch):
    """train_lora.py với batch lớn nhất vừa VRAM; resume đúng cấu hình nếu đã có train_shape.json."""
    shape_file = Path(output_dir) / "train_shape.json"
    if shape_file.exists():
        shape = json.loads(shape_file.read_text())
        return sh(cmd + ["--batch-size", shape["batch_size"], "--gradient-accumulation-steps",
                         shape["gradient_accumulation_steps"],
                         "--gradient-checkpointing" if shape["gradient_checkpointing"] else "--no-gradient-checkpointing"])
    attempts = [(b, False) for b in batch_sizes] + [(min(batch_sizes), True)]
    for index, (batch, checkpointing) in enumerate(attempts):
        print(f"▶ Thử batch {batch} x accum {max(1, effective_batch // batch)}, checkpointing {'bật' if checkpointing else 'tắt'}")
        args = ["--batch-size", batch, "--gradient-accumulation-steps", max(1, effective_batch // batch),
                "--gradient-checkpointing" if checkpointing else "--no-gradient-checkpointing"]
        if sh(cmd + args, check=index == len(attempts) - 1) == 0:
            return 0
        shape_file.unlink(missing_ok=True)
        print("⚠️ Lỗi (thường do thiếu VRAM) -> thử cấu hình nhẹ hơn.")


# Drive, code, thư viện
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
if (WORK / ".git").exists():
    for args in (["fetch", "--quiet", "origin", C["BRANCH"]], ["checkout", "--quiet", "-B", C["BRANCH"], f"origin/{C['BRANCH']}"]):
        subprocess.run(["git", "-C", str(WORK), *args], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", C["BRANCH"], C["REPO_URL"], str(WORK)], check=True)
os.chdir(WORK)
print("Code:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
if subprocess.run([sys.executable, "-c", "import vllm, peft"], capture_output=True).returncode != 0:
    sh("pip install -q vllm")
    sh("pip install -q peft datasets accelerate huggingface_hub wandb")
sh("pip uninstall -y -q torchaudio torchao", check=False)   # bản Colab không tương thích torch của vllm

# Dữ liệu + split
for name in ["human_train.json", "auto_train.json", "human_public_test.json", "auto_public_test.json"]:
    if not (WORK / "Data" / name).exists():
        shutil.copy2(Path(C["DRIVE_DATA_DIR"]) / name, WORK / "Data" / name)
if not (WORK / "Data/splits/auto_dev.json").exists():
    sh("python make_splits.py --from-manifest")

# Resume sau khi Colab ngắt (predictions_e4), rồi mọi dự đoán + parses của notebook E3 (predictions_e3)
for folder in ("predictions_e4", "predictions_e3"):
    root = DRIVE_OUT / folder
    for path in root.rglob("*.jsonl") if root.exists() else []:
        target = PRED / path.relative_to(root)
        if not target.exists() and not path.name.endswith(".raw.jsonl") and "LP_smoke" not in path.parts:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
PRED.mkdir(parents=True, exist_ok=True)

# HF + W&B
os.environ.setdefault("HF_TOKEN", secret("HF_TOKEN"))
if not os.environ.get("HF_USER"):
    os.environ["HF_USER"] = C.get("HF_USER") or ""
if not os.environ["HF_USER"] and os.environ.get("HF_TOKEN"):
    from huggingface_hub import HfApi
    os.environ["HF_USER"] = HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"]
if C["USE_WANDB"] and not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY")
os.environ.setdefault("WANDB_PROJECT", C["WANDB_PROJECT"])
os.environ["WANDB_LOG_MODEL"] = "false"
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"

In [ ]:
need = ["LP/human_train.parses.jsonl", "LP/human_dev.parses.jsonl", "LP/human_test.parses.jsonl",
        "LPT/human_dev.jsonl", "F1/human_dev.jsonl", "F2C/human_dev.jsonl", "LCOT/human_dev.jsonl", "S/auto_dev.jsonl"]
missing = [n for n in need if not (PRED / n).exists()]
assert not missing, f"Thiếu {missing}: chạy notebook run_colab_e3.ipynb trước (nó sao lưu vào Drive outputs/predictions_e3)."
(OUT / "e4").mkdir(parents=True, exist_ok=True)
print("Có sẵn:", sorted(str(p.relative_to(PRED)) for p in PRED.rglob("*.jsonl") if ".raw." not in p.name))

## 3. Phần 1 (CPU): LP+ trên Human train
3a: grid các công tắc mới (cố định quy ước E3). 3b: bảng gần/xa FR (C1), so sánh ngoài fold. 3c: ghi nguồn **LPX** cho train/dev/test.

In [ ]:
from IPython.display import Markdown, display


if C["RUN_LP_PLUS"]:
    t0 = time.time()
    sh(["python", "solve_parsed.py", "--parses", PRED / "LP/human_train.parses.jsonl", "--input", split_file("human", "train"),
        *sum((["--set", s] for s in C["LP_BASE"]), []), "--grid",
        "--grid-keys", "forms_mode,relax,definite,fr_block_share,world_merge", "--best-json", OUT / "e4/lp_best.json"])
    best = json.loads((OUT / "e4/lp_best.json").read_text())
    base = best["settings"]                 # gồm cả quy ước E3 (LP_BASE)
    print("Quy ước tốt nhất trên train:", base, f"(mean {best['mean']:.4f})")

    # C1: bảng gần/xa, chấm ngoài fold trên train rồi mới quyết định có dùng không
    import re
    def mean_of(text): return float(re.findall(r"mean \(abstain = wrong\): ([0-9.]+)", text)[-1])
    def run_text(cmd):
        result = subprocess.run(cmd, cwd=WORK, capture_output=True, text=True)
        if result.returncode:
            print(result.stdout[-3000:], result.stderr[-3000:])
            raise RuntimeError(f"Lệnh lỗi (exit {result.returncode}): {' '.join(map(str, cmd))}")
        return result.stdout
    common = ["python", "solve_parsed.py", "--parses", str(PRED / "LP/human_train.parses.jsonl"),
              "--input", split_file("human", "train"), *sum((["--set", s] for s in base), [])]
    without = mean_of(run_text(common))
    with_oof = mean_of(run_text(common + ["--oof-calibration", "5"]))
    use_calibration = with_oof > without + 1e-9
    print(f"C1 trên train (ngoài fold): không bảng {without:.4f} | có bảng {with_oof:.4f} -> {'DÙNG' if use_calibration else 'KHÔNG dùng'}")
    cal = OUT / "e4/fr_calibration.json"
    sh(common + ["--fit-fr-calibration", str(cal)])

    # 3c: nguồn LPX (train: bảng học ngoài fold; dev/test: bảng học trên toàn bộ train)
    extra_cal = ["--fr-calibration", str(cal)] if use_calibration else []
    sh(common + (["--oof-calibration", "5"] if use_calibration else []) +
       ["--source", "LPX", "--output", str(PRED / "LPX/human_train.jsonl")])
    for split in ("dev", "test"):
        sh(["python", "solve_parsed.py", "--parses", PRED / f"LP/human_{split}.parses.jsonl", "--input", split_file("human", split),
            *sum((["--set", s] for s in base), []), *extra_cal, "--source", "LPX", "--output", PRED / f"LPX/human_{split}.jsonl"])
    # LPT trên train (quy ước E3) để so sánh công bằng ở phần 3
    if not (PRED / "LPT/human_train.jsonl").exists():
        sh(["python", "solve_parsed.py", "--parses", PRED / "LP/human_train.parses.jsonl", "--input", split_file("human", "train"),
            "--set", "cross_block_far=true", "--source", "LPT", "--output", PRED / "LPT/human_train.jsonl"])
    (OUT / "e4/lp_choice.json").write_text(json.dumps({"settings": base, "calibration": use_calibration,
                                                       "train_mean_no_cal": without, "train_mean_oof_cal": with_oof}, indent=1))
    print(f"Phần 1 xong sau {(time.time() - t0) / 60:.1f} phút")
    to_drive("predictions_e4"); to_drive("e4")

### 3d. Dev: LP (E3) so với LPX, từng nguồn và trong tổ hợp (tune trên dev như trước)

In [ ]:
from spartqa.data import read_json
from spartqa.inference import fallback_answers, score_predictions
from spartqa.postprocess import PostprocessOptions
from spartqa.predictions import read_predictions

OPTIONS = PostprocessOptions(use_indifinite=C["USE_INDIFINITE"], human_yn_dk=C["HUMAN_YN_DK"])
FALLBACK = fallback_answers(Path("Data/splits"), "human")
rows = ["| Nguồn | Tập | YN | FR | FB | CO | TB |", "|---|---|---|---|---|---|---|"]
for source in ("LPT", "LPX"):
    for split in ("train", "dev"):
        path = PRED / source / f"human_{split}.jsonl"
        if path.exists():
            r = score_predictions(read_json(split_file("human", split)), read_predictions(path), "human", OPTIONS, FALLBACK)
            m = lambda t: r["by_task"][t]["accuracy" if t in ("YN", "CO") else "exact_match"]
            rows.append(f"| {source} | {split} | " + " | ".join(f"{m(t):.3f}" for t in ("YN", "FR", "FB", "CO"))
                        + f" | **{r['primary_macro_unofficial']:.3f}** |")
display(Markdown("\n".join(rows)))
(OUT / "e4/lp_sources.md").write_text("\n".join(rows) + "\n", encoding="utf-8")
names = ",".join(f"{d}={n}" for d, n in C["SUBMISSION_NAMES"].items())
for tag, branches in (("dev_LPT", "F=F2C,F1 S=S L=LCOT P=LPT"), ("dev_LPX", "F=F2C,F1 S=S L=LCOT P=LPX")):
    sh(["python", "compare_branches.py", "--pred-dir", PRED, "--output-dir", OUT / "e4" / tag, "--branches", branches,
        "--human-yn-dk", C["HUMAN_YN_DK"], "--use-indifinite" if C["USE_INDIFINITE"] else "--no-use-indifinite",
        "--submission-names", names, "--run-name", f"E4-compare-{tag}"])
to_drive("e4")

## 4. Phần 2 (GPU): dự đoán ngoài mẫu trên Human train
4a: F1 và S trên train. 4b: L trên train (các câu YN/FB/CO còn thiếu). 4c: cross-fit F2C, 5 fold.

In [ ]:
IF, L, FC = C["INFER_F"], C["LCOT"], C["F2C"]
infer_args = ["--max-model-len", IF["max_model_len"], "--max-num-seqs", IF["max_num_seqs"],
              "--gpu-memory-utilization", IF["gpu_memory_utilization"], "--seed", C["SEED"]]
f1_local = CKPT / "f_auto/adapter"
F1_ADAPTER = str(f1_local) if (f1_local / "adapter_config.json").exists() else (
    C["HF_F_AUTO_REPO"] if "/" in C["HF_F_AUTO_REPO"] else f"{os.environ['HF_USER']}/{C['HF_F_AUTO_REPO']}")
T = {}

t0 = time.time()
if C["RUN_F1_TRAIN"] and not (PRED / "F1/human_train.jsonl").exists():
    sh(["python", "infer_f.py", "--adapter", F1_ADAPTER, "--source", "F1", "--run-name", "E4-F1-human-train",
        *jobs("F1", [("human", "train")]), *infer_args, "--cache-dir", "/content/hf_adapters"])
if not (PRED / "S/human_train.jsonl").exists():
    sh(["python", "solve_symbolic.py", *jobs("S", [("human", "train")]), "--run-name", "E4-S-human-train"])
T["F1+S train"] = time.time() - t0; to_drive("predictions_e4")

t0 = time.time()
if C["RUN_LCOT_TRAIN"]:
    base_cmd = ["python", "llm_cot.py", "--source", "LCOT", "--n", L["n"], "--max-tokens", L["max_tokens"],
                "--max-model-len", L["max_model_len"], "--max-num-seqs", L["max_num_seqs"], "--fewshot-k", L["fewshot_k"],
                "--gpu-memory-utilization", L["gpu_memory_utilization"], "--seed", C["SEED"], "--fallback-model", "",
                "--run-name", "E4-LCOT-human-train", *jobs("LCOT", [("human", "train")])]
    gpu("trước L train")
    if sh(base_cmd + ["--model-name", L["model"]], check=False) != 0:
        print(f"⚠️ {L['model']} không chạy được -> thử {L['fallback']}")
        sh(base_cmd + ["--model-name", L["fallback"]])
T["L train"] = time.time() - t0; to_drive("predictions_e4")
print({k: f"{v / 60:.1f} phút" for k, v in T.items()})

In [ ]:
t0 = time.time()
if C["RUN_CROSSFIT"]:
    sh(["python", "crossfit.py", "make-folds", "--folds", FC["folds"], "--seed", C["SEED"], "--output", "Data/crossfit"])
    for k in range(FC["folds"]):
        fold_pred = PRED / "F2C_folds" / f"fold{k}.jsonl"
        if fold_pred.exists():
            print(f"fold{k}: đã có dự đoán, bỏ qua"); continue
        tk = time.time()
        drive_adapter = CKPT / "crossfit" / f"fold{k}" / "adapter"
        local = Path(f"/content/ckpt_crossfit/fold{k}")
        if not (drive_adapter / "adapter_config.json").exists():
            sft = OUT / "sft" / f"crossfit_fold{k}"
            sh(["python", "prepare_sft.py", "--stage", "human", "--splits-dir", f"Data/crossfit/fold{k}",
                "--human-repeat", FC["human_repeat"], "--auto-mix", FC["auto_mix"], "--augment-converse", FC["augment"],
                "--seed", C["SEED"], "--output-dir", sft])
            gpu(f"trước train fold{k}")
            cmd = ["python", "train_lora.py", "--train-file", sft / "train.jsonl", "--dev-file", sft / "dev.jsonl",
                   "--model-name", C["F_MODEL"], "--init-adapter", F1_ADAPTER, "--output-dir", local,
                   "--stage", f"stage2-converse-fold{k}", "--dataset-tag", "human", "--epochs", FC["epochs"],
                   "--learning-rate", FC["learning_rate"], "--save-strategy", "epoch", "--max-len", FC["max_len"],
                   "--seed", C["SEED"], "--run-name", f"E4-F2C-fold{k}-train", "--no-push"]
            train(cmd, local, FC["batch_sizes"], FC["effective_batch"])
            shutil.copytree(local / "adapter", drive_adapter, dirs_exist_ok=True)
        sh(["python", "infer_f.py", "--adapter", drive_adapter, "--source", "F2C", "--run-name", f"E4-F2C-fold{k}-infer",
            "--job", f"human,Data/crossfit/fold{k}/heldout.json,{fold_pred}", *infer_args])
        to_drive("predictions_e4")
        print(f"✔ fold{k} xong sau {(time.time() - tk) / 60:.1f} phút")
    sh(["python", "crossfit.py", "merge", "--folds-dir", "Data/crossfit", "--pattern", str(PRED / "F2C_folds" / "fold{k}.jsonl"),
        "--source", "F2C", "--output", PRED / "F2C/human_train.jsonl"])
    to_drive("predictions_e4")
T["cross-fit F2C"] = time.time() - t0
print({k: f"{v / 60:.1f} phút" for k, v in T.items()})

## 5. Phần 3: tune ensemble trên train + dev (613 câu, 37 story), cv 5-fold theo story
Mọi nguồn đều ngoài mẫu trên train: F1 (chỉ train Auto), F2C (cross-fit), L và LP (không train). S không đọc được Human.
Riêng quy ước LP và bảng C1 được chọn trên train, nên LP/LPX trên train hơi lạc quan: bảng C1 đã được học ngoài fold, nhưng quy ước thì chọn trên chính train.

In [ ]:
t0 = time.time()
if C["RUN_COMPARE"]:
    sh(["python", "crossfit.py", "combine", "--pred-dir", PRED, "--name", "trdev"])
    VARIANTS = {"F+S+L (không LP)": "F=F2C,F1 S=S L=LCOT", "+ LP (E3)": "F=F2C,F1 S=S L=LCOT P=LPT",
                "+ LPX (E4)": "F=F2C,F1 S=S L=LCOT P=LPX"}
    summary = ["| Biến thể | Tune trên | Tổ hợp tốt nhất | Human cv | Auto cv | **Final cv** | YN | FR | FB | CO |",
               "|---|---|---|---|---|---|---|---|---|---|"]
    for variant, branches in VARIANTS.items():
        for split in ("trdev", "dev"):
            out_dir = OUT / "e4" / f"{split}_{variant.split()[1] if variant.startswith('+') else 'noLP'}"
            if split == "dev" and not (out_dir / "best.json").exists():
                sh(["python", "compare_branches.py", "--pred-dir", PRED, "--output-dir", out_dir, "--branches", branches,
                    "--human-yn-dk", C["HUMAN_YN_DK"], "--use-indifinite" if C["USE_INDIFINITE"] else "--no-use-indifinite",
                    "--submission-names", names, "--run-name", f"E4-{out_dir.name}"])
            elif split == "trdev":
                sh(["python", "compare_branches.py", "--pred-dir", PRED, "--output-dir", out_dir, "--branches", branches,
                    "--tune-splits", "human=trdev", "--human-yn-dk", C["HUMAN_YN_DK"],
                    "--use-indifinite" if C["USE_INDIFINITE"] else "--no-use-indifinite",
                    "--submission-names", names, "--run-name", f"E4-{out_dir.name}"])
            best = json.loads((out_dir / "best.json").read_text())
            result = json.loads((out_dir / "comparison.json").read_text())[best["combination"]]
            human, auto = result.get("human"), result.get("auto")
            fmt = lambda v: f"{v:.3f}" if v is not None else "n/a"
            per = human["cv"] if human else {}
            summary.append(f"| {variant} | {split} ({'613' if split == 'trdev' else '116'} câu) | {best['combination']} | "
                           f"{fmt(human and human['cv_mean'])} | {fmt(auto and auto['cv_mean'])} | **{best['final_cv']:.4f}** | "
                           + " | ".join(fmt(per.get(t, {}).get("accuracy" if t in ("YN", "CO") else "exact_match"))
                                        for t in ("YN", "FR", "FB", "CO")) + " |")
    text = "\n".join(summary)
    display(Markdown(text))
    (OUT / "e4/summary.md").write_text(text + "\n\nLP+: " + (OUT / "e4/lp_choice.json").read_text() + "\n", encoding="utf-8")
T["compare"] = time.time() - t0
to_drive("e4"); to_drive("predictions_e4")
print({k: f"{v / 60:.1f} phút" for k, v in T.items()})
print(f"Xong sau {(time.time() - START) / 60:.1f} phút. Bảng: {OUT / 'e4/summary.md'}; file nộp: outputs/e4/<biến thể>/<tổ hợp>/")